In [1]:
import sys
import torch

print("Python:", sys.version)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))


Python: 3.11.0 (main, Oct 24 2022, 18:26:48) [MSC v.1933 64 bit (AMD64)]
Torch version: 2.5.1+cu121
CUDA available: True
Device: cuda


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

In [3]:
DATA_PATH = r"C:\Users\admin\Desktop\Forex_algo\code\Data\EUR_USD_H1.parquet"

df = pd.read_parquet(DATA_PATH)

df.head(), df.tail(), (df["time"].min(), df["time"].max(), df.shape)


(                       time  volume    mid_o    mid_h    mid_l    mid_c  \
 0 2016-01-07 00:00:00+00:00     542  1.07764  1.07832  1.07744  1.07778   
 1 2016-01-07 01:00:00+00:00    3167  1.07776  1.08100  1.07748  1.08029   
 2 2016-01-07 02:00:00+00:00    1567  1.08026  1.08176  1.07996  1.08152   
 3 2016-01-07 03:00:00+00:00     914  1.08156  1.08257  1.08150  1.08187   
 4 2016-01-07 04:00:00+00:00     649  1.08190  1.08256  1.08156  1.08236   
 
      bid_o    bid_h    bid_l    bid_c    ask_o    ask_h    ask_l    ask_c  
 0  1.07757  1.07823  1.07735  1.07770  1.07772  1.07840  1.07752  1.07787  
 1  1.07768  1.08092  1.07740  1.08020  1.07784  1.08109  1.07756  1.08038  
 2  1.08018  1.08169  1.07987  1.08144  1.08035  1.08184  1.08005  1.08159  
 3  1.08147  1.08249  1.08142  1.08178  1.08164  1.08265  1.08157  1.08196  
 4  1.08182  1.08247  1.08147  1.08228  1.08199  1.08264  1.08163  1.08245  ,
                            time  volume    mid_o    mid_h    mid_l    mid_c  \

In [4]:
df = df.copy()

# Use mid close as main price
df["close"] = df["mid_c"]

# Candle structure
df["body"] = (df["mid_c"] - df["mid_o"]) / df["close"]
df["upper_wick"] = (df["mid_h"] - df["mid_c"]).abs() / df["close"]
df["lower_wick"] = (df["mid_o"] - df["mid_l"]).abs() / df["close"]
df["range"] = (df["mid_h"] - df["mid_l"]) / df["close"]

# Returns (momentum on different horizons)
df["ret_1"]  = df["close"].pct_change(1)
df["ret_3"]  = df["close"].pct_change(3)
df["ret_6"]  = df["close"].pct_change(6)
df["ret_12"] = df["close"].pct_change(12)
df["ret_24"] = df["close"].pct_change(24)

# Volatility (standard deviation of returns)
df["vol_10"] = df["ret_1"].rolling(10).std()
df["vol_20"] = df["ret_1"].rolling(20).std()

# Moving averages (trend)
df["ma_20"]  = df["close"].rolling(20).mean()
df["ma_50"]  = df["close"].rolling(50).mean()
df["ma_100"] = df["close"].rolling(100).mean()

# Distance from MAs (normalized)
df["dist_ma20"]  = df["close"] / df["ma_20"]  - 1
df["dist_ma50"]  = df["close"] / df["ma_50"]  - 1
df["dist_ma100"] = df["close"] / df["ma_100"] - 1

df = df.dropna().reset_index(drop=True)
df.head()


,time,volume,mid_o,mid_h,mid_l,mid_c,bid_o,bid_h,bid_l,bid_c,...,ret_12,ret_24,vol_10,vol_20,ma_20,ma_50,ma_100,dist_ma20,dist_ma50,dist_ma100
0,2016-01-13 03:00:00+00:00,450,1.08290,1.08324,1.08247,1.08250,1.08282,1.08316,1.08240,1.08243,...,0.000120,-0.005055,0.000719,0.001012,1.084551,1.086925,1.087161,-0.001892,-0.004071,-0.004287
1,2016-01-13 04:00:00+00:00,239,1.08247,1.08314,1.08243,1.08280,1.08239,1.08307,1.08235,1.08272,...,-0.000674,-0.004990,0.000611,0.001015,1.084269,1.086739,1.087211,-0.001354,-0.003624,-0.004057
2,2016-01-13 05:00:00+00:00,518,1.08279,1.08352,1.08272,1.08294,1.08272,1.08344,1.08266,1.08287,...,-0.000923,-0.004266,0.000593,0.000862,1.084132,1.086569,1.087237,-0.001099,-0.003340,-0.003953
3,2016-01-13 06:00:00+00:00,355,1.08290,1.08379,1.08290,1.08332,1.08283,1.08373,1.08282,1.08324,...,-0.001622,-0.004658,0.000620,0.000855,1.084054,1.086407,1.087255,-0.000678,-0.002842,-0.003620
4,2016-01-13 07:00:00+00:00,1647,1.08332,1.08366,1.08134,1.08162,1.08324,1.08358,1.08126,1.08155,...,-0.003528,-0.006987,0.000721,0.000900,1.083859,1.086204,1.087253,-0.002065,-0.004220,-0.005181


In [5]:
HORIZON = 12          # 12 hours into the future
TH = 0.0012           # about 12 pips threshold

# Future return
df["future_close"] = df["close"].shift(-HORIZON)
df["future_ret"] = df["future_close"] / df["close"] - 1

# Direction labels
df["direction"] = np.where(
    df["future_ret"] > TH, 1,
    np.where(df["future_ret"] < -TH, 0, np.nan)
)

df = df.dropna().reset_index(drop=True)

df["direction"].value_counts(normalize=True)


direction
1.0    0.50157
0.0    0.49843
Name: proportion, dtype: float64

In [6]:
feature_cols = [
    # candle structure
    "body", "upper_wick", "lower_wick", "range",
    # returns
    "ret_1", "ret_3", "ret_6", "ret_12", "ret_24",
    # volatility
    "vol_10", "vol_20",
    # moving averages
    "ma_20", "ma_50", "ma_100",
    # distance from MAs
    "dist_ma20", "dist_ma50", "dist_ma100",
    # you can add volume if you like
    "volume"
]

X_raw = df[feature_cols].values.astype(np.float32)
y_raw = df["direction"].values.astype(np.int64)

X_raw.shape, y_raw.shape


((38212, 18), (38212,))

In [7]:
from sklearn.preprocessing import StandardScaler

SEQ_LEN = 150  # 150-hour context

# 1) Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# 2) Turn into sequences
def create_sequences(X, y, seq_len):
    Xs, ys = [], []
    for i in range(seq_len, len(X)):
        Xs.append(X[i-seq_len:i])
        ys.append(y[i])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.int64)

X_seq, y_seq = create_sequences(X_scaled, y_raw, SEQ_LEN)
X_seq.shape, y_seq.shape


((38062, 150, 18), (38062,))

In [8]:
N = len(X_seq)
n_train = int(N * 0.7)
n_val   = int(N * 0.15)
n_test  = N - n_train - n_val

X_train = X_seq[:n_train]
y_train = y_seq[:n_train]

X_val   = X_seq[n_train:n_train+n_val]
y_val   = y_seq[n_train:n_train+n_val]

X_test  = X_seq[n_train+n_val:]
y_test  = y_seq[n_train+n_val:]

N, n_train, n_val, n_test


(38062, 26643, 5709, 5710)

In [9]:
class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

batch_size = 128

train_ds = SeqDataset(X_train, y_train)
val_ds   = SeqDataset(X_val, y_val)
test_ds  = SeqDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False)


In [10]:
class CNN1D(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=n_features, out_channels=32, kernel_size=5, padding=2)
        self.bn1   = nn.BatchNorm1d(32)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.bn2   = nn.BatchNorm1d(64)
        self.pool  = nn.AdaptiveAvgPool1d(1)
        self.fc1   = nn.Linear(64, 32)
        self.dropout = nn.Dropout(0.5)
        self.fc2   = nn.Linear(32, 2)  # 2 classes: up/down

    def forward(self, x):
        # x: (batch, seq_len, features) -> (batch, features, seq_len)
        x = x.permute(0, 2, 1)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool(x).squeeze(-1)      # (batch, 64)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

n_features = X_seq.shape[2]
model = CNN1D(n_features).to(device)
model


CNN1D(
  (conv1): Conv1d(18, 32, kernel_size=(5,), stride=(1,), padding=(2,))
  (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=(2,))
  (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool): AdaptiveAvgPool1d(output_size=1)
  (fc1): Linear(in_features=64, out_features=32, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=32, out_features=2, bias=True)
)

In [11]:
# Class weights to deal with imbalance
class_weights_np = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=y_train
)
class_weights = torch.tensor(class_weights_np, dtype=torch.float32, device=device)
print("Class weights:", class_weights)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


Class weights: tensor([0.9940, 1.0061], device='cuda:0')


In [12]:
def evaluate(loader):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total = 0
    with torch.no_grad():
        for Xb, yb in loader:
            Xb = Xb.to(device)
            yb = yb.to(device)
            out = model(Xb)
            loss = criterion(out, yb)
            total_loss += loss.item() * yb.size(0)
            preds = out.argmax(dim=1)
            total_correct += (preds == yb).sum().item()
            total += yb.size(0)
    return total_loss / total, total_correct / total

n_epochs = 20

for epoch in range(1, n_epochs+1):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for Xb, yb in train_loader:
        Xb = Xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        out = model(Xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * yb.size(0)
        preds = out.argmax(dim=1)
        train_correct += (preds == yb).sum().item()
        train_total += yb.size(0)

    val_loss, val_acc = evaluate(val_loader)
    print(f"Epoch {epoch:02d} | "
          f"Train Loss: {train_loss/train_total:.4f}, Train Acc: {train_correct/train_total:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")


Epoch 01 | Train Loss: 0.6942, Train Acc: 0.5085 | Val Loss: 0.6926, Val Acc: 0.5246
Epoch 02 | Train Loss: 0.6913, Train Acc: 0.5230 | Val Loss: 0.6928, Val Acc: 0.5265
Epoch 03 | Train Loss: 0.6905, Train Acc: 0.5269 | Val Loss: 0.6938, Val Acc: 0.4715
Epoch 04 | Train Loss: 0.6899, Train Acc: 0.5312 | Val Loss: 0.6947, Val Acc: 0.4763
Epoch 05 | Train Loss: 0.6884, Train Acc: 0.5356 | Val Loss: 0.7022, Val Acc: 0.4850
Epoch 06 | Train Loss: 0.6869, Train Acc: 0.5458 | Val Loss: 0.6956, Val Acc: 0.4573
Epoch 07 | Train Loss: 0.6857, Train Acc: 0.5496 | Val Loss: 0.7000, Val Acc: 0.4754
Epoch 08 | Train Loss: 0.6831, Train Acc: 0.5612 | Val Loss: 0.7024, Val Acc: 0.4822
Epoch 09 | Train Loss: 0.6807, Train Acc: 0.5669 | Val Loss: 0.7094, Val Acc: 0.4994
Epoch 10 | Train Loss: 0.6782, Train Acc: 0.5708 | Val Loss: 0.6962, Val Acc: 0.4908
Epoch 11 | Train Loss: 0.6755, Train Acc: 0.5769 | Val Loss: 0.6991, Val Acc: 0.4987
Epoch 12 | Train Loss: 0.6722, Train Acc: 0.5821 | Val Loss: 0.70

In [13]:
model.eval()
all_preds = []
all_true  = []

with torch.no_grad():
    for Xb, yb in test_loader:
        Xb = Xb.to(device)
        yb = yb.to(device)
        out = model(Xb)
        preds = out.argmax(dim=1)
        all_preds.append(preds.cpu().numpy())
        all_true.append(yb.cpu().numpy())

all_preds = np.concatenate(all_preds)
all_true  = np.concatenate(all_true)

print(classification_report(all_true, all_preds))
print(confusion_matrix(all_true, all_preds))


              precision    recall  f1-score   support

           0       0.49      0.92      0.64      2814
           1       0.51      0.08      0.14      2896

    accuracy                           0.49      5710
   macro avg       0.50      0.50      0.39      5710
weighted avg       0.50      0.49      0.39      5710

[[2575  239]
 [2652  244]]
